In [2]:
import os
import json
import pandas as pd
import glob
from pathlib import Path

# =========================================================
# ⚙️ 配置区域 (请根据您的实际情况修改这里)
# =========================================================

# 1. 结果保存的根目录
RESULTS_ROOT = r"results\dual_source_experiment_0125" 

# 2. 指定要读取的 Run ID 名称 (支持部分匹配，例如 "run_000")
#    脚本会查找所有包含此字符串的文件夹
TARGET_RUN_NAME = "run_001" 

# =========================================================
# 🛠️ 核心处理函数
# =========================================================

def get_experiment_data(root_path, target_run_name):
    """
    遍历目录，提取符合 target_run_name 的实验数据
    """
    data_records = []
    
    # 使用 glob 递归查找所有匹配 target_run_name 的目录
    # 假设结构: .../TSFM/Encoder/Corrector/Run_ID
    search_pattern = os.path.join(root_path, "**", target_run_name + "*")
    run_dirs = glob.glob(search_pattern, recursive=True)
    
    print(f"📂 在 '{root_path}' 中找到了 {len(run_dirs)} 个匹配 '{target_run_name}' 的实验记录。")

    for run_dir in run_dirs:
        if not os.path.isdir(run_dir):
            continue
            
        run_path = Path(run_dir)
        
        # --- 1. 解析目录结构获取元数据 ---
        # 假设层级结构是: root / TSFM / Encoder / Corrector_Arch / Run_ID
        # parent = Corrector_Arch
        # parent.parent = Encoder
        # parent.parent.parent = TSFM
        
        try:
            corrector_arch = run_path.parent.name
            encoder_name = run_path.parent.parent.name
            tsfm_name = run_path.parent.parent.parent.name
            
            # 如果您的目录层级不同，请在这里调整 parent 的层数
        except IndexError:
            print(f"⚠️ 跳过路径解析失败的目录: {run_path}")
            continue

        # --- 2. 读取 metrics.csv ---
        metrics_path = run_path / "metrics.csv"
        best_smape_imp = None
        
        if metrics_path.exists():
            try:
                df = pd.read_csv(metrics_path)
                # 获取最大的 Avg_sMAPE_Imp (代表该次实验的最佳性能)
                if "Avg_sMAPE_Imp" in df.columns:
                    best_smape_imp = df["Avg_sMAPE_Imp"].max()
                else:
                    print(f"⚠️ {run_path} 的 metrics.csv 中没有 Avg_sMAPE_Imp 列")
            except Exception as e:
                print(f"❌ 读取 CSV 失败 {metrics_path}: {e}")
        else:
            print(f"⚠️ 找不到 metrics.csv: {run_path}")

        # --- 3. 读取 hyperparams.json ---
        hp_path = run_path / "hyperparams.json"
        hp_info = {}
        if hp_path.exists():
            try:
                with open(hp_path, 'r', encoding='utf-8') as f:
                    hp_info = json.load(f)
            except Exception as e:
                print(f"❌ 读取 JSON 失败 {hp_path}: {e}")

        # --- 4. 收集记录 ---
        if best_smape_imp is not None:
            data_records.append({
                "TSFM": tsfm_name,
                "Encoder": encoder_name,
                "Corrector": corrector_arch,
                "RunID": run_path.name,
                "sMAPE_Imp": best_smape_imp,
                "Hyperparams": hp_info
            })
            
    return pd.DataFrame(data_records)

def format_hyperparams(hp_dict):
    """将超参数字典格式化为字符串，用于标题"""
    if not hp_dict:
        return "Unknown Params"
    
    # 提取关键参数进行展示
    keys = ["corrector_arch", "hidden_dim", "top_k", "n_layer", "n_head", "batch_size", "learning_rate"]
    info = []
    for k in keys:
        if k in hp_dict:
            info.append(f"{k}={hp_dict[k]}")
    return ", ".join(info)

# =========================================================
# 📊 执行与绘图
# =========================================================

# 1. 获取数据
df = get_experiment_data(RESULTS_ROOT, TARGET_RUN_NAME)

if not df.empty:
    # 2. 按 Corrector 架构分组（通常同一张表对比相同的 Corrector）
    unique_correctors = df["Corrector"].unique()

    for arch in unique_correctors:
        sub_df = df[df["Corrector"] == arch]
        
        # 提取第一行的超参数作为表标题（假设同一次 Grid Search 参数相同）
        first_hp = sub_df.iloc[0]["Hyperparams"]
        title_str = f"Model: {arch} | {format_hyperparams(first_hp)}"
        
        # 3. 创建透视表 (Rows=Encoder, Cols=TSFM, Values=sMAPE_Imp)
        pivot_table = sub_df.pivot_table(
            index="Encoder", 
            columns="TSFM", 
            values="sMAPE_Imp",
            aggfunc="max" # 如果有重复，取最大值
        )
        
        # 4. 样式美化
        print(f"\n{'='*80}")
        print(f"📋 {title_str}")
        print(f"{'='*80}")
        
        # 使用 Pandas Style 添加热力图背景
        styled_table = pivot_table.style.background_gradient(
            cmap="RdYlGn", # 红-黄-绿 颜色映射
            vmin=0,        # 假设最小改进为 0 (或者设为 None 自动适应)
            vmax=pivot_table.max().max()
        ).format("{:.2f}%").set_caption(title_str) # 保留两位小数
        
        # 在 Jupyter 中显示
        from IPython.display import display
        display(styled_table)
        
else:
    print("❌ 未找到任何有效数据，请检查路径配置。")

📂 在 'results\dual_source_experiment_0125' 中找到了 8 个匹配 'run_001' 的实验记录。

📋 Model: dual_fusion_large | corrector_arch=dual_source_fusion, top_k=50, batch_size=128, learning_rate=0.0001


TSFM,chronos_bolt_tiny,kairos_10m,moirai_small,tirex_base
Encoder,,,,
advanced_hybrid_math,0.97%,1.56%,-1.43%,-1.55%
units,-0.02%,-4.07%,-0.54%,-0.91%
